In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import AgentState

# Unlike the state in 2.2_runtime_context, we cannot set default when inheriting from AgentState
# We use AgentState to make the agent learn about something, e.g. favourite colour, during the conversation
# We use AgentState to make the agent keep track of something
# There's something called middleware also, which may be something that also provides state dynamically - I think we'll look at this in further modules
class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [ ]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

# Like with runtime context, we have to use the ToolRuntime to gain access to the state information
# Normally a tool just returns a string, which becomes a ToolMessage in the conversation. That’s all it can affect. But your tool needs to write favourite_colour into the custom agent state (CustomState), so other tools and the model can read them later. A plain return value can’t do that; Command(update=...) can.
# The "messages" entry is required. When a tool returns a Command() call, the framework no longer auto-generates the ToolMessage for you. The model issued a tool call and every tool call must be answered by a ToolMessage with the matching tool_call_id, or the next LLM call fails with an error. That’s why you build one manually with runtime.tool_call_id.
@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "ollama:gemma4",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [7]:
from pprint import pprint

# You'll see 'favourite_colour': 'green' at the start of the response dictionary
# You'll also see in the tool call for update_favourite_colour in the response
pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='ffdf6202-cb6f-4e67-8c67-ed8060a04111'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4', 'created_at': '2026-10-14T21:49:58.891368Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7905025250, 'load_duration': 3844378959, 'prompt_eval_count': 84, 'prompt_eval_duration': 357069000, 'eval_count': 108, 'eval_duration': 3700789000, 'logprobs': None, 'model_name': 'gemma4', 'model_provider': 'ollama'}, id='lc_run--01a13c64-a407-70b3-b7e8-349a1473649d-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'abf0147b-d4bd-4f3c-9984-728a7ca66595', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 84, 'output_tokens': 108, 'total_tokens': 192}),
              ToolMessage(content='Successfully updated favourite colour', name='updat

In [8]:

# We can also update the value within the CustomState (of favourite_colour) by invoking the agent, similar to how it was done in 2.2_runtime_context (context=ColourContext(favourite_colour="green"))
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='4ccb6843-d3fb-4dc8-ac96-ba0056db92ff'),
              AIMessage(content="I'm doing well, thank you for asking! I'm ready to help you with any questions or tasks you have.\n\nHow are you doing today? 😊", additional_kwargs={}, response_metadata={'model': 'gemma4', 'created_at': '2026-10-14T21:52:35.568319Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9099649958, 'load_duration': 2860708, 'prompt_eval_count': 85, 'prompt_eval_duration': 379666000, 'eval_count': 247, 'eval_duration': 8690397000, 'logprobs': None, 'model_name': 'gemma4', 'model_provider': 'ollama'}, id='lc_run--01a13c67-0360-7691-ac03-6f3e9a23c241-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 85, 'output_tokens': 247, 'total_tokens': 332})]}


## Read state

In [9]:
# We can read the state after we define something in it, this is also done via a tool!
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    "ollama:gemma4",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [10]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is red")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'red',
 'messages': [HumanMessage(content='My favourite colour is red', additional_kwargs={}, response_metadata={}, id='8d2ba052-30b8-41e8-8ee8-7a861417ac81'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4', 'created_at': '2026-10-14T21:56:24.633261Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9253966583, 'load_duration': 5663791, 'prompt_eval_count': 119, 'prompt_eval_duration': 471342000, 'eval_count': 225, 'eval_duration': 8767600000, 'logprobs': None, 'model_name': 'gemma4', 'model_provider': 'ollama'}, id='lc_run--01a13c6a-8190-7eb0-a82a-543351eec143-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'red'}, 'id': '31419807-7a81-4723-9eee-993f06ac0268', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 119, 'output_tokens': 225, 'total_tokens': 344}),
              ToolMessage(content='Successfully updated favourite colour', name='update_favou

In [11]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'red',
 'messages': [HumanMessage(content='My favourite colour is red', additional_kwargs={}, response_metadata={}, id='8d2ba052-30b8-41e8-8ee8-7a861417ac81'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4', 'created_at': '2026-10-14T21:56:24.633261Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9253966583, 'load_duration': 5663791, 'prompt_eval_count': 119, 'prompt_eval_duration': 471342000, 'eval_count': 225, 'eval_duration': 8767600000, 'logprobs': None, 'model_name': 'gemma4', 'model_provider': 'ollama'}, id='lc_run--01a13c6a-8190-7eb0-a82a-543351eec143-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'red'}, 'id': '31419807-7a81-4723-9eee-993f06ac0268', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 119, 'output_tokens': 225, 'total_tokens': 344}),
              ToolMessage(content='Successfully updated favourite colour', name='update_favou